<a href="https://colab.research.google.com/github/Decoding-Data-Science/nov25/blob/main/new-Langchain_chains_agent_15thov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Langchain setup

In [ ]:
!pip install openai==0.28.1 langchain==0.0.270 langchain-community

In [8]:
#secret Key

import os
from google.colab import userdata

# Retrieve API keys from Colab's secure storage

openai_api_key = userdata.get("openai")

# Set them as environment variables

if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

## Chains

Restaraunt Business Generator

In [ ]:
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.chains.sequential import SequentialChain
from langchain.chat_models import ChatOpenAI # Import ChatOpenAI

# Use ChatOpenAI with a chat-optimized model like gpt-3.5-turbo
llm = ChatOpenAI(temperature = 1, model_name="gpt-3.5-turbo")

prompt_template_name= PromptTemplate(
    input_variables = ['cuisine'],
    template = "i want to open a restaraunt for  {cuisine} food. Suggest a fancy name for this"
                                         )


name_chain=LLMChain(llm=llm,prompt=prompt_template_name,output_key="restaraunt_name")




# Re-initialize llm with ChatOpenAI for the second chain as well
llm = ChatOpenAI(temperature = 1, model_name="gpt-3.5-turbo")
prompt_template_items = PromptTemplate(input_variables=['restaraunt_name', 'cuisine'],
                                      template="Suggest me some menu items for {restaraunt_name} which is {cuisine} cuisine. Only mention items names ,break down according to category like drinks soup starters main course"
                                       )

food_items_chain=LLMChain(llm=llm,prompt=prompt_template_items,output_key="menu_items")




chain = SequentialChain(
    chains = [name_chain, food_items_chain],
    input_variables =['cuisine'],
    output_variables = ['restaraunt_name','menu_items']
)


In [ ]:
chain({"cuisine": "Indian"})

{'cuisine': 'Indian',
 'restaraunt_name': '"Spice Palace"',
 'menu_items': 'Sure! Here are some menu items for Spice Palace, an Indian cuisine restaurant:\n\nDrinks:\n1. Mango Lassi\n2. Masala Chai\n3. Rose Falooda\n\nSoup:\n1. Mulligatawny Soup\n2. Dal Shorba\n3. Tomato Rasam\n\nStarters:\n1. Samosas\n2. Pakoras\n3. Tandoori Chicken\n4. Chaat\n\nMain Course:\n1. Butter Chicken\n2. Rogan Josh\n3. Paneer Tikka Masala\n4. Chana Masala\n5. Lamb Biryani\n6. Aloo Gobi'}

In [ ]:
chain({"cuisine": "Italian"})

{'cuisine': 'Italian',
 'restaraunt_name': '"La Dolce Vita Trattoria"',
 'menu_items': 'Drinks:\n1. Limoncello Spritz\n2. Aperol Spritz\n3. Negroni\n4. Bellini\n\nSoup:\n1. Minestrone\n2. Zuppa di Pesce\n\nStarters:\n1. Bruschetta al Pomodoro\n2. Arancini\n3. Caprese Salad\n4. Calamari Fritti\n\nMain Course:\n1. Spaghetti Carbonara\n2. Risotto ai Funghi\n3. Lasagna alla Bolognese\n4. Osso Buco\n5. Saltimbocca alla Romana'}

In [ ]:
pip install gradio

In [ ]:
import gradio as gr
from langchain.chains import LLMChain, SequentialChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

# DDS Logo URL
DDS_LOGO_URL = "https://raw.githubusercontent.com/Decoding-Data-Science/airesidency/main/dds_logo.jpg"

# Define LLM chains as provided
llm = OpenAI(temperature=0.9)
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template="i want to open a restaraunt for  {cuisine} food. Suggest a fancy name for this"
)
name_chain = LLMChain(llm=llm, prompt=prompt_template_name, output_key="restaraunt_name")

llm = OpenAI(temperature=0.9)
prompt_template_items = PromptTemplate(
    input_variables=['restaraunt_name'],
    template="Suggest me some menu items for {restaraunt_name} which is {cuisine} cuisine. Only mention items names ,break down according to category like drinks soup starters main course"
)
food_items_chain = LLMChain(llm=llm, prompt=prompt_template_items, output_key="menu_items")

chain = SequentialChain(
    chains=[name_chain, food_items_chain],
    input_variables=['cuisine'],
    output_variables=['restaraunt_name', 'menu_items']
)

# Function to run the chain and return outputs
def generate_restaurant(cuisine):
    outputs = chain({'cuisine': cuisine})
    return outputs['restaraunt_name'], outputs['menu_items']

# Build Gradio interface
with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:
    with gr.Row():
        with gr.Column(scale=1, min_width=200):
            gr.Image(DDS_LOGO_URL, label="Decoding Data Science", elem_id="dds-logo")
            gr.Markdown("**Restaurant Business Generator**\nChoose a cuisine type and generate a restaurant name and menu instantly.")
            cuisine_input = gr.Textbox(label="Cuisine Type", placeholder="e.g. Italian, Japanese, Mexican")
            generate_btn = gr.Button("Generate")
        with gr.Column(scale=2):
            name_output = gr.Textbox(label="Suggested Restaurant Name", interactive=False)
            menu_output = gr.Textbox(label="Suggested Menu Items", lines=10, interactive=False)

    generate_btn.click(fn=generate_restaurant, inputs=cuisine_input, outputs=[name_output, menu_output])

if __name__ == "__main__":
    demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10aa41488cfa87c593.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:


import gradio as gr
from langchain.chains import LLMChain, SequentialChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

# DDS Logo URL
DDS_LOGO_URL = "https://raw.githubusercontent.com/Decoding-Data-Science/airesidency/main/dds_logo.jpg"

# Define LLM chains as provided
llm = OpenAI(temperature=0.9)
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template="i want to open a restaraunt for  {cuisine} food. Suggest a fancy name for this"
)
name_chain = LLMChain(llm=llm, prompt=prompt_template_name, output_key="restaraunt_name")

llm = OpenAI(temperature=0.9)
prompt_template_items = PromptTemplate(
    input_variables=['restaraunt_name', 'cuisine'],
    template="Suggest me some menu items for {restaraunt_name} which is {cuisine} cuisine. Only mention items names, break down according to category like drinks, soup, starters, main course"
)
food_items_chain = LLMChain(llm=llm, prompt=prompt_template_items, output_key="menu_items")

chain = SequentialChain(
    chains=[name_chain, food_items_chain],
    input_variables=['cuisine'],
    output_variables=['restaraunt_name', 'menu_items']
)

# Popular cuisines dropdown options
TOP_CUISINES = [
    "Italian", "Chinese", "Japanese", "Mexican", "Indian",
    "French", "Thai", "Mediterranean", "American", "Spanish"
]

# Function to run the chain and return outputs
def generate_restaurant(cuisine):
    outputs = chain({'cuisine': cuisine})
    return outputs['restaraunt_name'], outputs['menu_items']

# Build Gradio interface
with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:
    with gr.Row():
        with gr.Column(scale=1, min_width=200):
            gr.Image(DDS_LOGO_URL, label="Decoding Data Science", elem_id="dds-logo")
            gr.Markdown(
                "**Restaurant Business Generator**  \n"
                "Choose a cuisine type, or select from popular options, then generate a restaurant name and menu instantly."
            )
            cuisine_dropdown = gr.Dropdown(choices=TOP_CUISINES, label="Popular Cuisines", value=TOP_CUISINES[0])
            generate_btn = gr.Button("Generate")
        with gr.Column(scale=2):
            name_output = gr.Textbox(label="Suggested Restaurant Name", interactive=False)
            menu_output = gr.Textbox(label="Suggested Menu Items", lines=10, interactive=False)

    generate_btn.click(fn=generate_restaurant, inputs=cuisine_dropdown, outputs=[name_output, menu_output])

if __name__ == "__main__":
    demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://39cad06263e247a64f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
from langchain.chains import LLMChain, SequentialChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI

# DDS Logo URL
DDS_LOGO_URL = "https://raw.githubusercontent.com/Decoding-Data-Science/airesidency/main/dds_logo.jpg"

# Define LLM chains as provided
llm = OpenAI(temperature=0.9)
prompt_template_name = PromptTemplate(
    input_variables=['cuisine'],
    template="i want to open a restaraunt for  {cuisine} food. Suggest a fancy name for this"
)
name_chain = LLMChain(llm=llm, prompt=prompt_template_name, output_key="restaraunt_name")

llm = OpenAI(temperature=0.9)
prompt_template_items = PromptTemplate(
    input_variables=['restaraunt_name', 'cuisine'],
    template="Suggest me some menu items for {restaraunt_name} which is {cuisine} cuisine. Only mention items names, break down according to category like drinks, soup, starters, main course"
)
food_items_chain = LLMChain(llm=llm, prompt=prompt_template_items, output_key="menu_items")

chain = SequentialChain(
    chains=[name_chain, food_items_chain],
    input_variables=['cuisine'],
    output_variables=['restaraunt_name', 'menu_items']
)

# Popular cuisines dropdown options
TOP_CUISINES = [
    "Italian", "Chinese", "Japanese", "Mexican", "Indian",
    "French", "Thai", "Mediterranean", "American", "Spanish"
]

# Function to run the chain and return outputs
def generate_restaurant(cuisine):
    outputs = chain({'cuisine': cuisine})
    return outputs['restaraunt_name'], outputs['menu_items']

# Wrapper to choose between dropdown and custom input
def on_generate(selected, custom):
    cuisine = custom.strip() if custom and custom.strip() else selected
    return generate_restaurant(cuisine)

# Build Gradio interface
with gr.Blocks(title="Restaurant Business Generator", theme="default") as demo:
    # Center-aligned app title
    gr.Markdown("<h1 style='text-align:center'>Restaurant Business Generator</h1>")
    with gr.Row():
        with gr.Column(scale=1, min_width=200):
            # Logo resized to 300px width
            gr.Image(DDS_LOGO_URL, label="Decoding Data Science", elem_id="dds-logo", width=300)
            gr.Markdown(
                "Choose a cuisine from popular options or type your own, then generate a restaurant name and menu instantly."
            )
            cuisine_dropdown = gr.Dropdown(choices=TOP_CUISINES, label="Popular Cuisines", value=TOP_CUISINES[0])
            cuisine_textbox = gr.Textbox(label="Or enter custom cuisine", placeholder="e.g. Vietnamese, Peruvian")
            generate_btn = gr.Button("Generate")
        with gr.Column(scale=2):
            name_output = gr.Textbox(label="Suggested Restaurant Name", interactive=False)
            menu_output = gr.Textbox(label="Suggested Menu Items", lines=10, interactive=False)

    generate_btn.click(fn=on_generate, inputs=[cuisine_dropdown, cuisine_textbox], outputs=[name_output, menu_output])

if __name__ == "__main__":
    demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://85f8efde5774aff455.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Agents

In [ ]:
import os
from google.colab import userdata

SERPAPI_API_KEY = userdata.get("SERP_API")

# Set them as environment variables

if SERPAPI_API_KEY:
    os.environ["SERPAPI_API_KEY"] = SERPAPI_API_KEY

In [ ]:
pip install google-search-results

#### serpapi and llm-math tool

In [ ]:
pip install wikipedia

In [ ]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI # Import ChatOpenAI

# Initialize LLM with a supported chat model
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["serpapi","wikipedia", "llm-math"], llm=llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

# Let's test it out!
agent.run("What was the GDP of US in 2024? Add 5 trillion to this number.")



> Entering new AgentExecutor chain...
I need to find the GDP of the US in 2024 first before adding 5 trillion to it.
Action: Search
Action Input: "US GDP 2024"
Observation: Real GDP increased 2.8 percent in 2024 (from the 2023 annual level to the 2024 annual level), the same as previously estimated. The increase in ...
Thought:I need to find the actual GDP number in 2024.
Action: Search
Action Input: "US GDP 2024 in trillions"
Observation: Global gross domestic product amounted to approximately 117.2 trillion U.S. dollars in 2025, with the United States making up more than one-quarter of this ...
Thought:I need to keep looking for the specific GDP number for the US in 2024.
Action: Search
Action Input: "US GDP in trillions 2024"
Observation: Current-dollar nominal GDP increased by 6.04 percent annualized, or $443.616 billion, to a level of $30.486 trillion, the current size of the U.S. economy.
Thought:I now have the GDP of the US in 2024, which is $30.486 trillion. Now I need to add

'$35.486 trillion'

In [ ]:
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI # Import ChatOpenAI

# Initialize LLM with a supported chat model
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["serpapi","wikipedia", "llm-math"], llm=llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

# Let's test it out!
#agent.run("what is the per capita income of Dubai? Use a search engine to find #the most recent data..")

agent.run("When was Elon musk born? What is his current age in days?")



> Entering new AgentExecutor chain...
I need to find out Elon Musk's birthdate first before calculating his current age in days.
Action: Wikipedia
Action Input: Elon Musk
Observation: Page: Elon Musk
Summary: Elon Reeve Musk (born June 28, 1971) is a businessman and entrepreneur known for his leadership of Tesla, SpaceX, X, and xAI. Musk has been the wealthiest person in the world since 2021; as of October 2025, Forbes estimates his net worth to be around $500 billion.
Born into a wealthy family in Pretoria, South Africa, Musk emigrated in 1989 to Canada; he has Canadian citizenship since his mother was born there. He received bachelor's degrees in 1997 from the University of Pennsylvania in Philadelphia, United States, before moving to California to pursue business ventures. In 1995, Musk co-founded the software company Zip2. Following its sale in 1999, he co-founded X.com, an online payment company that later merged to form PayPal, which was acquired by eBay in 2002. Musk also beca

ValueError: LLMMathChain._evaluate("
(datetime.today() - datetime(1971, 6, 28)).days
") raised error: Expression (datetime.today() - datetime(1971, 6, 28)).days has forbidden control characters.. Please try again with a valid numerical expression

#### Wikipedia and llm-math tool

In [ ]:
pip install wikipedia

In [ ]:
# install this package: pip install wikipedia

# The tools we'll give the Agent access to. Note that the 'llm-math' tool uses an LLM, so we need to pass that in.
tools = load_tools(["wikipedia", "llm-math"], llm=llm)

# Finally, let's initialize an agent with the tools, the language model, and the type of agent we want to use.
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Let's test it out!
#agent.run("When was Elon musk born? What is his current age in days")
agent.run("How many balloons can we fit in a380?")





> Entering new AgentExecutor chain...
 We need to find the volume of the a380 and the volume of a balloon to determine how many can fit.
Action: Calculator
Action Input: Volume of a380 = 845 m^3, Volume of a balloon = 0.014 m^3
Observation: Answer: 845.014
Thought: This is the total volume of the a380 and the balloon combined.
Action: Calculator
Action Input: Divide 845.014 by 0.014
Observation: Answer: 60358.142857142855
Thought: This is the maximum number of balloons that can fit in the a380.
Final Answer: 60358 balloons can fit in a380.

> Finished chain.


'60358 balloons can fit in a380.'

In [ ]:
pip install langchain_openai

In [ ]:
from langchain_openai import ChatOpenAI

# Using gpt-4
llm_gpt4 = ChatOpenAI(model="gpt-4")

In [ ]:
# pip install -q wikipedia google-search-results langchain langchain-openai

import os
from langchain_openai import ChatOpenAI
from langchain.agents import AgentType, initialize_agent, load_tools

# -------------------------------------------------------------------
# 0. API KEYS (set these in your env in practice, not hard-coded)
# -------------------------------------------------------------------
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
os.environ["SERPAPI_API_KEY"] = "YOUR_SERPAPI_API_KEY"

# -------------------------------------------------------------------
# 1. LLM: strong reasoning model for tool use
# -------------------------------------------------------------------
llm = ChatOpenAI(
    model="gpt-4.1-mini",   # or "gpt-4.1" / "gpt-4o" if you have access
    temperature=0,          # deterministic for reasoning + tools
)

# -------------------------------------------------------------------
# 2. Tools: SerpAPI + Wikipedia + LLM Math
# -------------------------------------------------------------------
tools = load_tools(
    ["serpapi", "wikipedia", "llm-math"],
    llm=llm,
    serpapi_api_key=os.environ["SERPAPI_API_KEY"],
)

# -------------------------------------------------------------------
# 3. Agent: structured ReAct-style agent for complex, multi-step queries
# -------------------------------------------------------------------
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,   # give some room for multi-step reasoning
    agent_kwargs={
        "system_message": (
            "You are a careful, analytical assistant.\n"
            "- Use SerpAPI for fresh or non-Wikipedia web information.\n"
            "- Use Wikipedia for well-known entities and background facts.\n"
            "- Use llm-math for any non-trivial calculations.\n"
            "For complex questions, break the problem into steps, "
            "use tools when needed, and explain your assumptions clearly."
        )
    },
)

# -------------------------------------------------------------------
# 4. Test query: A380 balloon Fermi estimation
# -------------------------------------------------------------------
query = (
    "How many standard party balloons (30 cm diameter) can fit inside an Airbus A380? "
    "Use realistic assumptions, fetch any needed dimensions, and show the estimation steps."
)

# Recommended (new-style) call:
response = agent.invoke({"input": query})
print(response["output"])

# Legacy style (still works if you prefer):
# answer = agent.run(query)
# print(answer)


ModuleNotFoundError: No module named 'langchain_core.memory'

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.agents import AgentType, initialize_agent, load_tools

# Make sure these are set in your environment in practice
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["SERPAPI_API_KEY"] = "YOUR_SERPAPI_API_KEY"

# 1. LLM: switch from old `OpenAI` to chat-based `ChatOpenAI`
llm = ChatOpenAI(
    model="gpt-4.1-mini",   # or "gpt-4.1" / "gpt-4o" if you have access
    temperature=0,          # stable, reasoning-focused
)

# 2. Tools: SerpAPI + Wikipedia + llm-math
tools = load_tools(
    ["serpapi", "wikipedia", "llm-math"],
    llm=llm,
    serpapi_api_key=os.environ.get("SERPAPI_API_KEY")
)

# 3. Agent: use a stronger structured agent for complex queries
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10,
    agent_kwargs={
        "system_message": (
            "You are a careful, up-to-date reasoning assistant.\n"
            "- Use SerpAPI first for current numerical data like per-capita income.\n"
            "- Use Wikipedia for background facts.\n"
            "- Use llm-math for any non-trivial calculations.\n"
            "Always explain your steps briefly and cite the year of any figures you report."
        )
    },
)

# 4. Test it out
query = "What is the per capita income of Dubai? Use a search engine to find th"


ModuleNotFoundError: No module named 'langchain_core.memory'

# Task
Upgrade `langchain`, `langchain-community`, and `langchain-openai` to their latest stable versions, and install `pypdf` and `chromadb`.

## Upgrade Langchain Libraries

### Subtask:
Upgrade `langchain`, `langchain-community`, and `langchain-openai` to their latest stable versions to resolve compatibility issues and the `ModuleNotFoundError`.


**Reasoning**:
The subtask requires upgrading specific LangChain libraries to their latest stable versions. This can be achieved using `pip install --upgrade`.



In [1]:
pip install --upgrade langchain langchain-community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.1.3
    Uninstalling langchain-core-1.1.3:
      Successfully uninstalled langchain-core-1.1.3
  Attempting uninstall: langchain
    Found existing installation: langchain 1.1.3
    Uninstalling lang

**Reasoning**:
The subtask also requires installing `pypdf` and `chromadb` to ensure all necessary libraries are present.



In [2]:
pip install pypdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.2/328.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 15.3 MB/s et

## Prepare Sample Document

### Subtask:
Provide a sample document (e.g., a PDF file or a text string) that the RAG system will use to answer questions.


A sample document will be provided as a multi-paragraph text string and then saved to a `.txt` file named `sample_document.txt` in the current Colab environment. This will serve as the knowledge base for the RAG system.

**Reasoning**:
Next, I will create a multi-paragraph text string and save it to a `.txt` file as specified in the instructions, preparing the sample document for the RAG system.



In [45]:
document_text = """ENERGY PLANT OPERATIONS NOTE
Document ID: OPS-PMP-017
Title: Troubleshooting Overheated Cooling Water Pump (P-201A)
Site: Al Noor Combined Cycle Power Plant
Unit: Block 2 | Area: Cooling Water System (CWS)
Last Updated: 2025-12-10
Owner: Maintenance Reliability Team

1) PURPOSE
This document provides a step-by-step procedure to diagnose and stabilize an overheated cooling water pump (P-201A). It is intended for control room operators, rotating equipment technicians, and reliability engineers.

2) EQUIPMENT OVERVIEW
- Asset Tag: P-201A (Cooling Water Pump – Duty)
- Type: Horizontal centrifugal pump, electric motor driven
- Normal Operating Range:
  - Discharge Pressure: 6.0–7.5 bar
  - Flow: 1,500–2,200 m3/h (depending on condenser demand)
  - Bearing Temperature (DE/NDE): 55–80°C typical (alarm at 90°C, trip at 100°C)
  - Motor Current: 120–165 A typical
- Instruments:
  - TT-201A-DE / TT-201A-NDE (bearing temp)
  - PT-201A-DIS (discharge pressure)
  - FT-201A (flow)
  - VT-201A (vibration monitoring, if installed)

3) INCIDENT TRIGGER (SYMPTOMS)
Typical event conditions reported in DCS:
- Alarm: “P-201A Bearing Temp High” (TT-201A-DE > 90°C)
- Operator notes:
  - Discharge pressure dropping or unstable
  - Abnormal vibration trend or noise near coupling
  - Seal flush line temperature elevated (if applicable)
  - Motor current rising above normal range

4) SAFETY FIRST (MANDATORY)
Before any field action:
- Follow plant LOTO procedure if opening guards, removing coupling cover, or working on energized equipment.
- Wear PPE: gloves, face shield, safety shoes, hearing protection.
- Hot surfaces hazard: bearing housings and casing can exceed safe touch temperature.
- If leak observed near mechanical seal, treat as pressurized water hazard.

5) QUICK STABILIZATION (FIRST 5 MINUTES)
Goal: prevent equipment damage and maintain plant cooling demand.
A. Confirm the alarm source:
   - Verify TT values for DE and NDE.
   - Cross-check with handheld IR thermometer if safe.
B. Reduce stress on the pump (operator action):
   - Verify pump is operating near its best efficiency point (BEP).
   - If suction pressure is low, check upstream strainers and basin level.
C. If temperature continues rising rapidly:
   - Prepare to start standby pump P-201B (if available).
   - Coordinate with Control Room Supervisor for controlled switchover.

6) ROOT CAUSE CHECKLIST (MOST COMMON)
A. Low suction / cavitation
- Evidence:
  - “Gravel” sound, fluctuating discharge pressure, vibration spikes
- Checks:
  - Cooling water basin level low
  - Suction strainer DP high (clogging)
  - Air ingress at suction flange / gasket
- Immediate action:
  - Clean strainer (per permit), restore basin level, verify suction valves fully open

B. Bearing lubrication issue
- Evidence:
  - DE bearing heats faster than NDE
  - Grease purge blocked or grease contaminated
- Checks:
  - Grease type correct per OEM?
  - Over-greasing can cause heat (too much grease churn)
- Action:
  - Follow OEM lubrication interval and quantity
  - If contamination suspected, plan bearing inspection

C. Misalignment / coupling issue
- Evidence:
  - Vibration increasing across 1X running speed
  - Coupling hot spot, noise, abnormal wear
- Action:
  - Schedule laser alignment check after switching to standby pump

D. Mechanical seal / seal flush failure
- Evidence:
  - Seal area hot, leakage rate changes, flush line blocked
- Checks:
  - Seal flush valve open? Flush flow present?
  - Flush strainer clogged?
- Action:
  - Restore flush flow, inspect flush strainer, log in CMMS

E. Blocked discharge / operation at low flow
- Evidence:
  - Discharge pressure high but flow low
  - Pump recirculation/min-flow line closed
- Action:
  - Verify min-flow / recirculation line per procedure
  - Avoid prolonged low-flow operation (overheating risk)

7) CONTROL ROOM DECISION: WHEN TO SWITCH TO STANDBY PUMP
Switch from P-201A to P-201B if any of the following:
- Bearing temperature > 95°C and rising for 3 minutes
- Vibration exceeds site limit (e.g., > 7.1 mm/s RMS)
- Motor current > 180 A with unstable discharge pressure
- Visible seal leak worsening or safety risk present

8) DOCUMENTATION & SYSTEMS (SAP / CMMS / DCS)
Log the event with:
- DCS Alarm Screenshot / Trend (TT, PT, FT, current)
- Operator logbook entry (time, alarm, actions taken)
- SAP PM Notification template:
  - Equipment: P-201A
  - Symptom: Overheating DE bearing temp high
  - Suspected cause: (choose from checklist)
  - Immediate action: (e.g., switched to P-201B, cleaned strainer)
  - Recommended follow-up: alignment check, bearing inspection, seal flush check

9) RECOMMENDED FOLLOW-UP WORK (NEXT 24–72 HOURS)
- Inspect suction strainer DP transmitter calibration
- Review vibration spectrum (if condition monitoring exists)
- Check lubrication records (grease type/quantity/interval)
- Verify pump curve vs operating point (ensure near BEP)
- Conduct coupling alignment and soft foot check
- If repeated events: perform RCA (5-Why) and update PM plan

10) KEY TERMS (FOR RAG SEARCH)
P-201A, pump overheating, bearing temperature high, cavitation, suction strainer DP, seal flush, misalignment, coupling heat, cooling water system, standby pump switchover, SAP PM notification, DCS alarm trend.
"""

with open("energy_plant_sample_document.txt", "w") as f:
    f.write(document_text)

print("✅ Sample document 'energy_plant_sample_document.txt' created successfully.")


✅ Sample document 'energy_plant_sample_document.txt' created successfully.


## Load and Split Document

### Subtask:
Implement code to load the document using a document loader and then split it into smaller, manageable chunks using a `RecursiveCharacterTextSplitter`.


**Reasoning**:
First, I will import the necessary classes, TextLoader and RecursiveCharacterTextSplitter, from their respective langchain libraries to prepare for document loading and splitting.



In [42]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Imported TextLoader and RecursiveCharacterTextSplitter.")

Imported TextLoader and RecursiveCharacterTextSplitter.


**Reasoning**:
Now that the necessary libraries are imported, I will initialize the TextLoader with the sample document, load it, initialize the RecursiveCharacterTextSplitter with specified chunk size and overlap, and then split the document into chunks, finally printing the number of generated chunks.



In [47]:
loader = TextLoader("energy_plant_sample_document.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(f"Number of generated chunks: {len(chunks)}")

Number of generated chunks: 15


In [48]:
documents

[Document(metadata={'source': 'energy_plant_sample_document.txt'}, page_content='ENERGY PLANT OPERATIONS NOTE\nDocument ID: OPS-PMP-017\nTitle: Troubleshooting Overheated Cooling Water Pump (P-201A)\nSite: Al Noor Combined Cycle Power Plant\nUnit: Block 2 | Area: Cooling Water System (CWS)\nLast Updated: 2025-12-10\nOwner: Maintenance Reliability Team\n\n1) PURPOSE\nThis document provides a step-by-step procedure to diagnose and stabilize an overheated cooling water pump (P-201A). It is intended for control room operators, rotating equipment technicians, and reliability engineers.\n\n2) EQUIPMENT OVERVIEW\n- Asset Tag: P-201A (Cooling Water Pump – Duty)\n- Type: Horizontal centrifugal pump, electric motor driven\n- Normal Operating Range:\n  - Discharge Pressure: 6.0–7.5 bar\n  - Flow: 1,500–2,200 m3/h (depending on condenser demand)\n  - Bearing Temperature (DE/NDE): 55–80°C typical (alarm at 90°C, trip at 100°C)\n  - Motor Current: 120–165 A typical\n- Instruments:\n  - TT-201A-DE / 

## Create Embeddings and Vector Store

### Subtask:
Generate vector embeddings for the document chunks using `OpenAIEmbeddings` and store these embeddings in a `Chroma` vector store, enabling efficient semantic search.


**Reasoning**:
First, I will import the necessary classes, OpenAIEmbeddings and Chroma, from their respective langchain libraries to prepare for creating embeddings and the vector store, as per the first instruction.



In [49]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

print("Imported OpenAIEmbeddings and Chroma.")

Imported OpenAIEmbeddings and Chroma.


**Reasoning**:
Now that the necessary classes are imported, I will initialize OpenAIEmbeddings and then create a Chroma vector store from the chunks using these initialized embeddings, storing the result in a variable named vectorstore, as per the remaining instructions.



**Reasoning**:
The previous cell failed because the OpenAI API key was not properly recognized. I will explicitly set the `OPENAI_API_KEY` environment variable before initializing `OpenAIEmbeddings` to ensure it is available in the current runtime.



In [50]:
import os
from google.colab import userdata

# Retrieve API key and set it as an environment variable
openai_api_key = userdata.get("openai")
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(chunks, embeddings)

print("Embeddings generated and stored in Chroma vector store.")

Embeddings generated and stored in Chroma vector store.


## Define RAG Function as a Langchain Tool

### Subtask:
Create a function that uses the vector store to retrieve relevant document chunks based on a query, and then uses an LLM to synthesize an answer. Wrap this function as a Tool that the Langchain agent can utilize.


**Reasoning**:
First, I will import the necessary classes, `ChatOpenAI` and `Tool`, to prepare for initializing the LLM and defining the RAG tool.



**Reasoning**:
The previous attempt to import `Tool` failed because its location changed in the updated Langchain library. I need to import `Tool` from `langchain_core.tools` instead of `langchain.tools` to resolve this `ImportError`.



In [52]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool

print("Imported ChatOpenAI and Tool.")

Imported ChatOpenAI and Tool.


**Reasoning**:
Now that the necessary classes are imported, I will initialize the ChatOpenAI model, create a retriever, define the `rag_func` to retrieve documents and synthesize an answer using the LLM, and finally wrap this function as a Langchain Tool, as per the remaining instructions.



In [53]:
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

retriever = vectorstore.as_retriever()

def rag_func(query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    docs_content = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""Based on the following context, answer the question.

Context:
{docs_content}

Question: {query}
Answer:"""
    response = llm.invoke(prompt)
    return response.content

rag_tool = Tool(
    name="Amazon Rainforest RAG Tool",
    func=rag_func,
    description="Useful for answering questions about the Amazon rainforest based on provided documents."
)

print("ChatOpenAI model initialized, retriever created, rag_func defined, and wrapped as a Langchain Tool.")

ChatOpenAI model initialized, retriever created, rag_func defined, and wrapped as a Langchain Tool.


## Integrate RAG Tool into Agent

### Subtask:
Modify the existing `initialize_agent` setup to include the newly created RAG tool alongside other tools like `SerpAPI` and `Wikipedia`. This allows the agent to decide when to use the document-accessing RAG tool.


**Reasoning**:
I will update the `initialize_agent` setup to include the `rag_tool` alongside the standard tools, ensuring the `SERPAPI_API_KEY` is correctly configured and using the specified agent type and parameters.



**Reasoning**:
The `ImportError` for `AgentType` indicates a change in its location within the Langchain library after the upgrade. I need to import `AgentType` from its new path, `langchain.agents.agent_types`, to resolve this.



**Reasoning**:
The `initialize_agent` function was removed in recent versions of Langchain, causing the `ImportError`. I will refactor the agent initialization to use the new `create_tool_calling_agent` and `AgentExecutor` pattern, which is the recommended approach for agents with tool-calling models like 'gpt-4o-mini'. This involves creating an explicit chat prompt template for the agent's system message and then combining the LLM, tools, and prompt into an `AgentExecutor` instance.



**Reasoning**:
The previous `ImportError` for `AgentExecutor` indicates that its import path has changed within the updated Langchain library. I will explicitly import `AgentExecutor` from `langchain.agents.agent` to resolve this issue and align with the new module structure.



**Reasoning**:
The previous `ImportError` for `load_tools` indicates its import path has changed within the updated Langchain library. I will explicitly import `load_tools` from `langchain_community.tools` to resolve this issue and align with the new module structure.



**Reasoning**:
The previous `ImportError` for `create_tool_calling_agent` indicates its import path has changed within the updated Langchain library. I will explicitly import `create_tool_calling_agent` from `langchain.agents.tool_calling_agent` to resolve this issue and align with the new module structure.



**Reasoning**:
The `ModuleNotFoundError` for `langchain.agents.tool_calling_agent` indicates that `create_tool_calling_agent` is no longer located there. I will correct its import path to `langchain.agents`, as is standard for the updated Langchain library.



**Reasoning**:
The persistent `ImportError` for `create_tool_calling_agent` from `langchain.agents` suggests that its location might have been moved to `langchain_community.agents` during the recent major library upgrade, similar to how `load_tools` was moved. I will update the import path to `langchain_community.agents` to resolve this.



In [56]:
!pip -q install -U "langchain>=0.2.0" "langchain-core>=0.2.0" "langchain-community>=0.2.0" "langchain-openai>=0.1.0"


**Reasoning**:
The consistent `ImportError` for `create_tool_calling_agent` from both `langchain.agents` and `langchain_community.agents`, coupled with a `ModuleNotFoundError` for `langchain.agents.tool_calling_agent`, suggests that the function's location has shifted to a more specific internal module. I will attempt to import `create_tool_calling_agent` from `langchain.agents.tool_calling` to align with possible internal refactoring in the upgraded library.



**Reasoning**:
The consistent `ModuleNotFoundError` for `langchain.agents.tool_calling` and previous `ImportError` for `create_tool_calling_agent` suggest varying import paths in recent Langchain versions. I will implement a robust `try-except` block to import `create_tool_calling_agent` from either `langchain.agents` or `langchain.agents.openai_functions`, and also correct the `placeholder` in the `ChatPromptTemplate` to use `MessagesPlaceholder` for compatibility with the updated agent API.



**Reasoning**:
The `ModuleNotFoundError` for `langchain.agents.agent` indicates that `AgentExecutor` is no longer located there. I will correct its import path to be directly from `langchain.agents`, and consolidate the `create_tool_calling_agent` import under the main `langchain.agents` module, as is standard for updated Langchain libraries.



# Task
Refactor the Langchain agent setup in cells `8b7a9d15` and `90CVA-ZlB_gS` to correct the RAG tool's name to 'Amazon_Rainforest_RAG_Tool' to comply with OpenAI API naming conventions, update imports for `create_tool_calling_agent` and `AgentExecutor`, and adjust the agent's invocation input format for compatibility with upgraded Langchain libraries.

## Correct RAG Tool Name

### Subtask:
Modify the `rag_tool` definition in cell `8b7a9d15` to change its `name` from 'Amazon Rainforest RAG Tool' to 'Amazon_Rainforest_RAG_Tool' to comply with OpenAI API naming conventions.


**Reasoning**:
To fix the `BadRequestError` related to the tool name, I will modify the `rag_tool` definition in cell `8b7a9d15` to change its `name` from 'Amazon Rainforest RAG Tool' to 'Amazon_Rainforest_RAG_Tool' to comply with OpenAI API naming conventions. I will regenerate the full cell content to ensure all dependencies are correctly defined and the change is applied.



In [57]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

retriever = vectorstore.as_retriever()

def rag_func(query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    docs_content = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""Based on the following context, answer the question.

Context:
{docs_content}

Question: {query}
Answer:"""
    response = llm.invoke(prompt)
    return response.content

rag_tool = Tool(
    name="RAG_Tool", # Corrected the name here
    func=rag_func,
    description="Useful for answering questions about the Amazon rainforest based on provided documents."
)

print("ChatOpenAI model initialized, retriever created, rag_func defined, and wrapped as a Langchain Tool with corrected name.")

ChatOpenAI model initialized, retriever created, rag_func defined, and wrapped as a Langchain Tool with corrected name.


**Reasoning**:
The previous code failed due to `ImportError` for `AgentExecutor` and the incorrect usage of `create_agent` from `langgraph`. I will now refactor the agent setup to use `create_tool_calling_agent` and `AgentExecutor` with their correct import paths, and adjust the prompt to use `MessagesPlaceholder` for compatibility with the upgraded Langchain libraries, addressing all remaining points of the subtask.



In [63]:
#!pip -q install -U langchain langchain-core langchain-community langchain-openai google-search-results wikipedia

import os
from google.colab import userdata
import importlib.metadata as md
import re

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools

print("langchain:", md.version("langchain"))
print("langchain-core:", md.version("langchain-core"))
print("langchain-openai:", md.version("langchain-openai"))
print("langchain-community:", md.version("langchain-community"))

# -------------------------
# Helpers: sanitize tool names
# -------------------------
_valid = re.compile(r"^[a-zA-Z0-9_-]+$")

def sanitize_tool_name(name: str) -> str:
    # replace spaces and illegal chars with underscore
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", name).strip("_")
    if not safe:
        safe = "tool"
    return safe

def sanitize_tools(tools):
    for t in tools:
        # LangChain tools typically have .name attribute
        if hasattr(t, "name"):
            original = t.name
            safe = sanitize_tool_name(original)
            t.name = safe
            if original != safe:
                print(f"🔧 Renamed tool: '{original}' -> '{safe}'")
        # Some tools may expose .description only; ignore
    return tools

# -------------------------
# Keys
# -------------------------
openai_key = userdata.get("openai")
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key

serpapi_key = userdata.get("SERPAPI_API_KEY") or userdata.get("SERP_API")
if serpapi_key:
    os.environ["SERPAPI_API_KEY"] = serpapi_key

# -------------------------
# LLM
# -------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -------------------------
# Standard tools
# -------------------------
standard_tools = load_tools(
    ["serpapi", "wikipedia", "llm-math"],
    llm=llm,
    serpapi_api_key=os.environ.get("SERPAPI_API_KEY"),
)

# -------------------------
# Your rag_tool must already exist
# -------------------------
# Example problematic name might be: "Amazon Rainforest RAG Tool"
# This will fix it automatically:
all_tools = standard_tools + [rag_tool]
all_tools = sanitize_tools(all_tools)

# Double-check tool names are valid
for i, t in enumerate(all_tools):
    nm = getattr(t, "name", None)
    if nm and not _valid.match(nm):
        raise ValueError(f"Tool at index {i} still has invalid name: {nm}")

# -------------------------
# Create agent (LangChain v1)
# -------------------------
system_prompt = (
    "You are a careful, analytical assistant.\n"
    "-  Use SerpAPI for fresh web info.\n"
    "- Use Wikipedia for background facts.\n"
    "- Use llm-math for calculations.\n"
    "- Use the RAG tool for questions about the provided documents.\n"
)

agent_graph = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=system_prompt,
    debug=True,
)

print("✅ Agent graph created.")

# -------------------------
# Test
# -------------------------
result = agent_graph.invoke(
    {"messages": [{"role": "user", "content": "tell me about P-201A Bearing Temp High"}]}
)

print("\nFINAL:\n", result["messages"][-1].content)

langchain: 1.2.0
langchain-core: 1.2.1
langchain-openai: 1.1.3
langchain-community: 0.4.1
✅ Agent graph created.
[values] {'messages': [HumanMessage(content='tell me about P-201A Bearing Temp High', additional_kwargs={}, response_metadata={}, id='97f30933-07e0-483e-bd3f-afc4e1040a5a')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 250, 'total_tokens': 269, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a460d7e2b7', 'id': 'chatcmpl-Cn7odDNKaNjhXxALsRGG8epyBwB2B', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b2352-3ac8-79b2-ac05-11bcea368d6b-0', tool_calls=[{'name': 'wik

ValueError: Got error from SerpAPI: Your account has run out of searches.

**Reasoning**:
The persistent `ModuleNotFoundError` for `AgentExecutor` and `create_tool_calling_agent` suggests that their import paths are still incorrect in the upgraded Langchain environment. I will try importing both directly from the top-level `langchain.agents` module, as this is the standard location for these classes in Langchain `0.2.x` and later versions. If `create_tool_calling_agent` still fails, I will use the `create_openai_tools_agent` fallback.

